# Transformer as Fourth Model

Adds a Transformer encoder as a fourth model alongside XGBoost, Deep NN, and LSTM.
Uses the exact same data pipeline, folds, and fairness evaluation as previous notebooks.

### What this notebook does:
1. Loads aligned data using cv_stay_ids.npy
2. Trains a Transformer encoder in 5-fold CV
3. Compares all four models on AUPRC by race group
4. Extracts attention weights to show what the model focuses on per demographic group
5. Runs the same statistical tests as thesis_imputation_auprc.py

### Input:
- cv_stay_ids.npy
- cohort.csv, X_ts.npy, M_ts.npy
- oof_probs_XGBoost.npy, oof_probs_Deep_NN.npy, oof_probs_LSTM.npy



---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import itertools
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve
)
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

OUT_DIR   = Path('/home/ino/thesis_outputs')
TRANS_DIR = OUT_DIR / 'transformer'
TRANS_DIR.mkdir(exist_ok=True)

VITAL_NAMES  = ['heart_rate', 'resp_rate', 'spo2', 'temperature', 'map']
KEEP_RACES   = ['White', 'Black', 'Hispanic', 'Asian']
ALL_MODELS   = ['XGBoost', 'Deep NN', 'LSTM', 'Transformer']
N_FOLDS      = 5
RANDOM_STATE = 42
N_BOOTSTRAP  = 2000
N_EPOCHS     = 60
BATCH_SIZE   = 512
LR           = 3e-4

COLORS = {
    'XGBoost':     'steelblue',
    'Deep NN':     'seagreen',
    'LSTM':        'darkorange',
    'Transformer': 'mediumpurple',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {device}  |  GPUs: {n_gpus}')
print(f'Output: {TRANS_DIR}')

---
## 2. Load & align data

In [ ]:
# Exact same alignment as all previous notebooks
cv_stay_ids = np.load(OUT_DIR / 'cv_stay_ids.npy')
print(f'CV stay_ids: {len(cv_stay_ids):,}')

cohort_full = pd.read_csv(OUT_DIR / 'cohort.csv')
cohort_full = cohort_full.set_index('stay_id')
cohort_cv   = cohort_full.loc[cv_stay_ids].reset_index()

cohort_full_reset = pd.read_csv(OUT_DIR / 'cohort.csv').reset_index(drop=True)
stay_id_to_ts_idx = {sid: i for i, sid in
                     enumerate(cohort_full_reset['stay_id'].values)}

X_ts_full  = np.load(OUT_DIR / 'X_ts.npy')
M_ts_full  = np.load(OUT_DIR / 'M_ts.npy')
ts_indices = np.array([stay_id_to_ts_idx[sid] for sid in cv_stay_ids])
X_ts_cv    = X_ts_full[ts_indices]
M_ts_cv    = M_ts_full[ts_indices]

# Filter out Other
keep_mask = cohort_cv['race_group'].isin(KEEP_RACES).values
cohort    = cohort_cv[keep_mask].reset_index(drop=True)
kept_idx  = np.where(keep_mask)[0]

X_ts    = X_ts_cv[kept_idx]   # (N, 24, 5)
M_ts    = M_ts_cv[kept_idx]   # (N, 24, 5)

y    = cohort['hospital_expire_flag'].values.astype(np.int32)
demo = cohort[['race_group', 'gender', 'age_group']].reset_index(drop=True)
race_arr = demo['race_group'].values

print(f'Cohort: {len(cohort):,}  |  Mortality: {y.mean():.1%}')
print(cohort['race_group'].value_counts().to_string())

# Load existing OOF probs
oof_probs = {
    'XGBoost': np.load(OUT_DIR / 'oof_probs_XGBoost.npy')[kept_idx],
    'Deep NN': np.load(OUT_DIR / 'oof_probs_Deep_NN.npy')[kept_idx],
    'LSTM':    np.load(OUT_DIR / 'oof_probs_LSTM.npy')[kept_idx],
}

print('\nExisting model AUPRC (sanity check):')
for m in ['XGBoost', 'Deep NN', 'LSTM']:
    print(f'  {m:<12}  AUPRC={average_precision_score(y, oof_probs[m]):.4f}')

---
## 3. Transformer model definition

Architecture:
- Input projection: 10 -> d_model (vital values + missingness mask)
- Learnable positional embeddings: one per hour (24 positions)
- 3-layer Transformer encoder with multi-head self-attention
- Mean pooling over the 24 time steps
- 2-layer MLP classifier

This is the same input format as the LSTM (batch, 24, 10).
The key difference is that attention is computed across ALL 24 hours
simultaneously, instead of sequentially as in the LSTM.
This lets the model learn relationships between any two time points
directly, regardless of distance.

In [ ]:
class MortalityTransformer(nn.Module):
    """
    Transformer encoder for ICU vital sign time series.
    Input:  (batch, 24, 10)  — 5 vitals + 5 missingness flags
    Output: (batch,)         — mortality logit
    """
    def __init__(self, input_size=10, d_model=128, nhead=8,
                 num_layers=3, dropout=0.3, max_len=24):
        super().__init__()

        # Project raw input to d_model dimensions
        self.input_proj = nn.Linear(input_size, d_model)

        # Learnable positional embedding — one per hour
        self.pos_embedding = nn.Embedding(max_len, d_model)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,   # Pre-LN for more stable training
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x, return_attn=False):
        # x: (batch, 24, 10)
        batch_size, seq_len, _ = x.shape
        positions = torch.arange(seq_len, device=x.device)

        # Embed and add positional encoding
        x = self.input_proj(x) + self.pos_embedding(positions)
        x = self.dropout(x)

        # Transformer encoder
        x = self.transformer(x)   # (batch, 24, d_model)

        # Mean pooling over time dimension
        x = x.mean(dim=1)         # (batch, d_model)

        return self.classifier(x).squeeze(1)


def get_attention_weights(model, X_input, batch_size=256):
    """
    Extract attention weights from all transformer layers.
    Returns mean attention weight per hour position averaged
    across heads and layers: shape (N, 24)
    """
    model.eval()
    all_attn = []

    # Hook to capture attention weights
    attn_weights = []
    hooks = []

    def make_hook(layer_idx):
        def hook(module, input, output):
            # TransformerEncoderLayer stores attn weights in output[1]
            # We need to re-run with need_weights=True
            pass
        return hook

    # Simpler approach: run with manual attention extraction
    with torch.no_grad():
        for s in range(0, len(X_input), batch_size):
            xb = torch.tensor(X_input[s:s+batch_size],
                              dtype=torch.float32).to(device)
            bs, seq_len, _ = xb.shape
            positions = torch.arange(seq_len, device=device)
            emb = model.input_proj(xb) + model.pos_embedding(positions)

            # Get attention from first encoder layer
            layer = model.transformer.layers[0]
            # norm_first=True so apply norm first
            normed = layer.norm1(emb)
            _, attn = layer.self_attn(
                normed, normed, normed,
                need_weights=True, average_attn_weights=True
            )
            # attn: (batch, 24, 24) — attention from each position to others
            # Sum over source positions to get importance of each hour
            attn_per_hour = attn.sum(dim=1)   # (batch, 24)
            all_attn.append(attn_per_hour.cpu().numpy())

    return np.concatenate(all_attn, axis=0)   # (N, 24)


print('Transformer model defined.')
# Quick parameter count
tmp = MortalityTransformer()
n_params = sum(p.numel() for p in tmp.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')
del tmp

---
## 4. Training infrastructure

In [ ]:
def train_transformer(model, train_loader, val_loader, y_val,
                      n_epochs=N_EPOCHS, lr=LR, fold=0):
    """
    Train transformer with cosine annealing LR and class weighting.
    Returns best out-of-fold probabilities.
    """
    # Class weight for imbalance
    pos_weight = torch.tensor(
        [(y_val == 0).sum() / (y_val == 1).sum()],
        dtype=torch.float32
    ).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-5)

    best_auprc = 0
    best_probs = None

    for epoch in range(1, n_epochs + 1):
        # Training
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # Validation
        model.eval()
        probs_list = []
        with torch.no_grad():
            for xb, _ in val_loader:
                probs_list.append(
                    torch.sigmoid(model(xb.to(device))).cpu().numpy())
        probs  = np.concatenate(probs_list)
        auprc  = average_precision_score(y_val, probs)

        if auprc > best_auprc:
            best_auprc = auprc
            best_probs = probs.copy()

        if epoch % 10 == 0 or epoch == 1:
            print(f'    Fold {fold+1} ep {epoch:02d}  '
                  f'AUPRC={auprc:.4f}{"  *" if auprc==best_auprc else ""}')

    print(f'    Fold {fold+1} best AUPRC={best_auprc:.4f}')
    return best_probs, best_auprc


print('Training infrastructure ready.')

---
## 5. Five-fold cross-validation

In [ ]:
# Build fold splits — identical seed to all previous notebooks
strat_label = (demo['race_group'].astype(str) + '_' +
               cohort['hospital_expire_flag'].astype(str)).values
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=RANDOM_STATE)
fold_splits  = list(skf.split(np.zeros(len(cohort)), strat_label))

print(f'Starting {N_FOLDS}-fold CV for Transformer...')
print(f'Epochs: {N_EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LR}')
print('=' * 60)

oof_transformer  = np.zeros(len(cohort))
fold_auprcs      = []
# Store attention weights for fairness analysis
oof_attn_weights = np.zeros((len(cohort), 24))

for fold, (train_idx, test_idx) in enumerate(fold_splits):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──────────────────────────────')

    # Prepare time series input (same as LSTM)
    X_tr_ts  = X_ts[train_idx]
    X_te_ts  = X_ts[test_idx]
    M_tr_ts  = M_ts[train_idx]
    M_te_ts  = M_ts[test_idx]
    y_tr     = y[train_idx]
    y_te     = y[test_idx]

    # Normalise using training fold statistics
    ts_mean  = X_tr_ts.mean(axis=(0,1), keepdims=True)
    ts_std   = X_tr_ts.std(axis=(0,1),  keepdims=True) + 1e-8
    X_tr_n   = (X_tr_ts - ts_mean) / ts_std
    X_te_n   = (X_te_ts - ts_mean) / ts_std

    # Concatenate vitals + missingness mask -> (N, 24, 10)
    X_tr_in  = np.concatenate([X_tr_n, M_tr_ts], axis=2).astype(np.float32)
    X_te_in  = np.concatenate([X_te_n, M_te_ts], axis=2).astype(np.float32)

    y_tr_t   = torch.tensor(y_tr, dtype=torch.float32)
    y_te_t   = torch.tensor(y_te, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_tr_in), y_tr_t),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_te_in), y_te_t),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True
    )

    # Build model
    model = MortalityTransformer(
        input_size=10, d_model=128, nhead=8,
        num_layers=3, dropout=0.3
    )
    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    # Train
    best_probs, best_auprc = train_transformer(
        model, train_loader, val_loader, y_te, fold=fold)

    oof_transformer[test_idx] = best_probs
    fold_auprcs.append(best_auprc)

    # Extract attention weights for test patients
    base_model = model.module if hasattr(model, 'module') else model
    oof_attn_weights[test_idx] = get_attention_weights(base_model, X_te_in)

    # Save model
    torch.save(base_model.state_dict(),
               TRANS_DIR / f'transformer_fold{fold+1}.pt')

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save OOF probs
np.save(OUT_DIR / 'oof_probs_Transformer.npy', oof_transformer)
np.save(TRANS_DIR / 'oof_attn_weights.npy', oof_attn_weights)
oof_probs['Transformer'] = oof_transformer

print('\n' + '='*60)
print('Transformer CV complete!')
print(f'Fold AUPRCs: {[f"{x:.4f}" for x in fold_auprcs]}')
print(f'Mean AUPRC: {np.mean(fold_auprcs):.4f} ± {np.std(fold_auprcs):.4f}')
print(f'OOF AUPRC:  {average_precision_score(y, oof_transformer):.4f}')

---
## 6. Overall performance of all 4 models

In [ ]:
print('Overall OOF performance — all four models:')
print(f'{"Model":<14} {"AUPRC":>8} {"AUROC":>8}')
print('-' * 35)
for m in ALL_MODELS:
    ap = average_precision_score(y, oof_probs[m])
    ar = roc_auc_score(y, oof_probs[m])
    print(f'{m:<14} {ap:>8.4f} {ar:>8.4f}')

# PR curves — overall
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

from sklearn.metrics import roc_curve
for m in ALL_MODELS:
    ap   = average_precision_score(y, oof_probs[m])
    ar   = roc_auc_score(y, oof_probs[m])
    fpr, tpr, _ = roc_curve(y, oof_probs[m])
    prec, rec, _ = precision_recall_curve(y, oof_probs[m])
    axes[0].plot(fpr, tpr, color=COLORS[m], linewidth=2,
                 label=f'{m} (AUROC={ar:.3f})')
    axes[1].plot(rec, prec, color=COLORS[m], linewidth=2,
                 label=f'{m} (AUPRC={ap:.3f})')

axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC curve — out-of-fold'); axes[0].legend(fontsize=9)

axes[1].axhline(y.mean(), color='gray', linestyle='--', alpha=0.5,
                label=f'Baseline ({y.mean():.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('PR curve — out-of-fold'); axes[1].legend(fontsize=9)

plt.suptitle('All four models — overall performance', fontsize=13)
plt.tight_layout()
plt.savefig(TRANS_DIR / 'fig_overall_performance_4models.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_overall_performance_4models.png')

---
## 7. AUPRC by race group of all four models

In [ ]:
# Point estimates
rows = []
for m in ALL_MODELS:
    probs = oof_probs[m]
    rows.append({'model': m, 'group': 'Overall',
                 'auprc': average_precision_score(y, probs),
                 'n': len(y)})
    for g in KEEP_RACES:
        mask = (demo['race_group'] == g).values
        yg, pg = y[mask], probs[mask]
        if yg.sum() < 5: continue
        rows.append({'model': m, 'group': g,
                     'auprc': average_precision_score(yg, pg),
                     'n': int(mask.sum())})

auprc_df = pd.DataFrame(rows)
print('AUPRC by race group (all four models):')
pivot = auprc_df[auprc_df['group']!='Overall'].pivot(
    index='group', columns='model', values='auprc').round(4)
print(pivot.to_string())
auprc_df.to_csv(TRANS_DIR / 'auprc_4models_by_group.csv', index=False)

# Disparity gap
print('\nAUPRC disparity gap (max - min across race groups):')
for m in ALL_MODELS:
    sub = auprc_df[(auprc_df['model']==m)&(auprc_df['group']!='Overall')]
    print(f'  {m:<14}  {sub["auprc"].max()-sub["auprc"].min():.4f}')

In [ ]:
# Bar chart of 4 models
x     = np.arange(len(KEEP_RACES))
width = 0.2
fig, ax = plt.subplots(figsize=(12, 5))

for offset, m in enumerate(ALL_MODELS):
    vals = [auprc_df[(auprc_df['model']==m)&
                     (auprc_df['group']==g)]['auprc'].values[0]
            for g in KEEP_RACES]
    ax.bar(x + offset*width, vals, width, label=m,
           color=COLORS[m], edgecolor='white', alpha=0.85)

for m in ALL_MODELS:
    ov = auprc_df[(auprc_df['model']==m)&
                  (auprc_df['group']=='Overall')]['auprc'].values[0]
    ax.axhline(ov, color=COLORS[m], linestyle='--', linewidth=1, alpha=0.5)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(KEEP_RACES)
ax.set_ylabel('AUPRC')
ax.set_title('AUPRC by race group — all four models (Other excluded)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(TRANS_DIR / 'fig_auprc_4models_by_race.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_auprc_4models_by_race.png')

In [ ]:
# PR curves per race group of all four models
fig, axes = plt.subplots(1, len(KEEP_RACES),
                         figsize=(5*len(KEEP_RACES), 4))
for ax, g in zip(axes, KEEP_RACES):
    mask = (demo['race_group'] == g).values
    yg   = y[mask]
    ax.axhline(yg.mean(), color='gray', linestyle='--', alpha=0.5,
               label=f'Baseline ({yg.mean():.2f})')
    for m in ALL_MODELS:
        pg = oof_probs[m][mask]
        prec, rec, _ = precision_recall_curve(yg, pg)
        ap = average_precision_score(yg, pg)
        ax.plot(rec, prec, color=COLORS[m], linewidth=1.8,
                label=f'{m} ({ap:.3f})')
    ax.set_title(f'{g}  (n={mask.sum():,})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend(fontsize=7)
plt.suptitle('PR curves by race group — all four models', fontsize=13)
plt.tight_layout()
plt.savefig(TRANS_DIR / 'fig_pr_curves_4models_by_race.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_pr_curves_4models_by_race.png')

---
## 8. Bootstrap CIs of all four models

In [ ]:
def bootstrap_auprc_ci(y_true, probs, n_boot=N_BOOTSTRAP, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys, ps = y_true[idx], probs[idx]
        if ys.sum() == 0 or ys.sum() == n: continue
        scores.append(average_precision_score(ys, ps))
    s = np.array(scores)
    return s.mean(), np.percentile(s, 2.5), np.percentile(s, 97.5)


print('Bootstrap 95% CIs — all four models...')
ci_records = []
for m in ALL_MODELS:
    probs = oof_probs[m]
    print(f'\n{m}:')
    mn, lo, hi = bootstrap_auprc_ci(y, probs)
    print(f'  {"Overall":<12}  {mn:.4f}  95% CI [{lo:.4f}, {hi:.4f}]')
    ci_records.append({'model': m, 'group': 'Overall',
                       'auprc': mn, 'ci_lower': lo, 'ci_upper': hi})
    for g in KEEP_RACES:
        mask = (demo['race_group'] == g).values
        yg, pg = y[mask], probs[mask]
        if yg.sum() < 10: continue
        mn, lo, hi = bootstrap_auprc_ci(yg, pg)
        print(f'  {g:<12}  {mn:.4f}  95% CI [{lo:.4f}, {hi:.4f}]')
        ci_records.append({'model': m, 'group': g,
                           'auprc': mn, 'ci_lower': lo, 'ci_upper': hi})

ci_df = pd.DataFrame(ci_records)
ci_df.to_csv(TRANS_DIR / 'auprc_bootstrap_ci_4models.csv', index=False)
print('\nSaved auprc_bootstrap_ci_4models.csv')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, m in zip(axes, ALL_MODELS):
    sub = ci_df[(ci_df['model']==m)&(ci_df['group']!='Overall')].sort_values('auprc')
    y_pos = np.arange(len(sub))
    ax.barh(y_pos, sub['auprc'], color=COLORS[m], alpha=0.75, height=0.5)
    ax.errorbar(sub['auprc'], y_pos,
                xerr=[sub['auprc']-sub['ci_lower'],
                      sub['ci_upper']-sub['auprc']],
                fmt='none', color='black', capsize=5, linewidth=1.8)
    overall = ci_df[(ci_df['model']==m)&
                    (ci_df['group']=='Overall')]['auprc'].values[0]
    ax.axvline(overall, color='crimson', linestyle='--', linewidth=1.5,
               label=f'Overall ({overall:.3f})')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['group'])
    ax.set_xlabel('AUPRC (95% bootstrap CI)')
    ax.set_title(m); ax.legend(fontsize=8)
plt.suptitle('AUPRC with 95% bootstrap CIs — all four models', fontsize=13)
plt.tight_layout()
plt.savefig(TRANS_DIR / 'fig_bootstrap_ci_4models.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_bootstrap_ci_4models.png')

---
## 9. Attention weight analysis


In [ ]:
attn = oof_attn_weights   # (N, 24)

# Mean attention per hour for each race group
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hours = np.arange(1, 25)
ax = axes[0]
for g in KEEP_RACES:
    mask = (demo['race_group'] == g).values
    mean_attn = attn[mask].mean(axis=0)
    ax.plot(hours, mean_attn, linewidth=2, label=g,
            color=list({'White':'#4878CF','Black':'#D65F5F',
                        'Hispanic':'#6ACC65','Asian':'#B47CC7'}.values())
            [KEEP_RACES.index(g)])
ax.set_xlabel('Hour of ICU stay')
ax.set_ylabel('Mean attention weight')
ax.set_title('Transformer attention by race group\n'
             'Which hours does the model focus on?')
ax.legend(fontsize=9)
ax.set_xticks(hours[::2])

# Difference: Black minus White attention
ax = axes[1]
mask_b = (demo['race_group'] == 'Black').values
mask_w = (demo['race_group'] == 'White').values
diff   = attn[mask_b].mean(axis=0) - attn[mask_w].mean(axis=0)
colors_bar = ['#D65F5F' if d > 0 else '#4878CF' for d in diff]
ax.bar(hours, diff, color=colors_bar, edgecolor='white', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Hour of ICU stay')
ax.set_ylabel('Attention difference (Black − White)')
ax.set_title('Differential attention: Black vs White\n'
             'Red = model attends more for Black patients')
ax.set_xticks(hours[::2])

plt.suptitle('Transformer attention weight analysis by race group', fontsize=13)
plt.tight_layout()
plt.savefig(TRANS_DIR / 'fig_attention_by_race.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_attention_by_race.png')

In [ ]:
# Attention heatmap of survivors vs non-survivors per race
fig, axes = plt.subplots(2, len(KEEP_RACES),
                         figsize=(5*len(KEEP_RACES), 6))

for col, g in enumerate(KEEP_RACES):
    mask  = (demo['race_group'] == g).values
    yg    = y[mask]
    attn_g = attn[mask]

    for row, (outcome, label) in enumerate([(0,'Survivors'), (1,'Non-survivors')]):
        ax      = axes[row, col]
        sub_idx = yg == outcome
        if sub_idx.sum() == 0:
            ax.axis('off'); continue
        mean_a  = attn_g[sub_idx].mean(axis=0, keepdims=True)  # (1,24)
        im = ax.imshow(mean_a, aspect='auto', cmap='YlOrRd',
                       vmin=0, vmax=attn.max())
        ax.set_xticks(np.arange(0, 24, 2))
        ax.set_xticklabels(np.arange(1, 25, 2), fontsize=8)
        ax.set_yticks([])
        ax.set_xlabel('Hour' if row==1 else '')
        ax.set_title(f'{g}\n{label}' if row==0 else label, fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Mean attention weight heatmap\n'
             'Survivors vs non-survivors by race group', fontsize=12)
plt.tight_layout()
plt.savefig(TRANS_DIR / 'fig_attention_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_attention_heatmap.png')

---
## 10. Statistical comparison of all four models

In [ ]:
def permutation_test_auprc(y_true, probs, mask_a, mask_b,
                            n_perm=N_BOOTSTRAP, seed=0):
    rng    = np.random.default_rng(seed)
    ya, pa = y_true[mask_a], probs[mask_a]
    yb, pb = y_true[mask_b], probs[mask_b]
    if ya.sum() == 0 or yb.sum() == 0:
        return np.nan, np.nan, np.nan, np.nan
    obs_a    = average_precision_score(ya, pa)
    obs_b    = average_precision_score(yb, pb)
    obs_diff = obs_a - obs_b
    y_pool   = np.concatenate([ya, yb])
    p_pool   = np.concatenate([pa, pb])
    na       = len(ya)
    perm_diffs = []
    for _ in range(n_perm):
        idx = rng.permutation(len(y_pool))
        ya_p, pa_p = y_pool[idx[:na]], p_pool[idx[:na]]
        yb_p, pb_p = y_pool[idx[na:]], p_pool[idx[na:]]
        if ya_p.sum() == 0 or yb_p.sum() == 0: continue
        perm_diffs.append(
            average_precision_score(ya_p, pa_p) -
            average_precision_score(yb_p, pb_p)
        )
    perm_diffs = np.array(perm_diffs)
    p_val = np.mean(np.abs(perm_diffs) >= np.abs(obs_diff))
    return obs_diff, obs_a, obs_b, p_val


print('Black vs White permutation test — all four models:')
print('=' * 65)
perm_records = []
for m in ALL_MODELS:
    probs  = oof_probs[m]
    mask_b = (demo['race_group'] == 'Black').values
    mask_w = (demo['race_group'] == 'White').values
    diff, ap_b, ap_w, pval = permutation_test_auprc(
        y, probs, mask_b, mask_w)
    sig = '(significant)' if pval < 0.05 else '(not significant)'
    print(f'  {m:<14}  Black={ap_b:.4f}  White={ap_w:.4f}  '
          f'Diff={diff:+.4f}  p={pval:.4f}  {sig}')
    perm_records.append({
        'model': m, 'auprc_black': ap_b,
        'auprc_white': ap_w, 'diff': diff, 'p_value': pval
    })

pd.DataFrame(perm_records).to_csv(
    TRANS_DIR / 'black_white_permtest_4models.csv', index=False)
print('\nSaved black_white_permtest_4models.csv')

---
## 11. Summary

In [ ]:
print('=' * 65)
print('FOUR-MODEL COMPARISON SUMMARY')
print('=' * 65)

print(f'\nCohort: {len(cohort):,}  |  Mortality: {y.mean():.1%}')

print('\n── Overall AUPRC ──')
for m in ALL_MODELS:
    ap = average_precision_score(y, oof_probs[m])
    ar = roc_auc_score(y, oof_probs[m])
    print(f'  {m:<14}  AUPRC={ap:.4f}  AUROC={ar:.4f}')

print('\n── AUPRC by race group ──')
pivot = auprc_df[auprc_df['group']!='Overall'].pivot(
    index='group', columns='model', values='auprc').round(4)
print(pivot[ALL_MODELS].to_string())

print('\n── Disparity gap (max - min AUPRC across race groups) ──')
for m in ALL_MODELS:
    sub = auprc_df[(auprc_df['model']==m)&(auprc_df['group']!='Overall')]
    print(f'  {m:<14}  {sub["auprc"].max()-sub["auprc"].min():.4f}')

print('\n── Black vs White AUPRC gap ──')
for r in perm_records:
    sig = '*' if r['p_value'] < 0.05 else ''
    print(f'  {r["model"]:<14}  gap={r["diff"]:+.4f}  '
          f'p={r["p_value"]:.4f}{sig}')

print(f'\nOutputs saved to: {TRANS_DIR}')
for f in sorted(TRANS_DIR.glob('*')):
    print(f'  {f.name}')